## Calculating the Hurst Exponent for India VIX and NIFTY50

In [5]:
import pandas as pd
import numpy as np
import yfinance as yf

In [4]:
def hurst_exponent(ts):
    """
    Compute Hurst exponent using R/S analysis.
    H < 0.5 → mean-reverting
    H ≈ 0.5 → random walk
    H > 0.5 → trending
    """
    lags = range(2, 20)
    tau = []
    for lag in lags:
        # std of lagged differences
        tau.append(np.std(np.subtract(ts[lag:], ts[:-lag])))
        
    # Fit log-log regression: slope = H
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0]


def rolling_hurst(series, window=120):
    return series.rolling(window).apply(lambda x: hurst_exponent(x.values), raw=False)

vix = yf.Ticker("^INDIAVIX").history(period="2y")["Close"].dropna()
vix.index = vix.index.tz_localize(None)

# Rolling Hurst
window = 120
vix_hurst = rolling_hurst(vix, window=window)
vix_rank = vix.rolling(252).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
latest_vix    = vix.iloc[-1]
latest_hurst  = vix_hurst.iloc[-1]
latest_rank   = vix_rank.iloc[-1]

print(f"India VIX        : {latest_vix:.2f}")
print(f"VIX Percentile   : {latest_rank:.2%}")
print(f"Hurst Exponent   : {latest_hurst:.3f}")

if latest_rank > 0.5 and latest_hurst < 0.5:
    print("Premium is rich AND vol is mean-reverting: Sell")
elif latest_rank > 0.5 and latest_hurst >= 0.5:
    print("Premium is rich BUT vol is trending: trade smaller / wait")
elif latest_rank <= 0.5:
    print("Premium is thin: avoid or skip")

India VIX        : 25.01
VIX Percentile   : 98.81%
Hurst Exponent   : 0.483
Premium is rich AND vol is mean-reverting: Sell


## Add Realised Volatiltiy Cones